In [1]:
import os

In [2]:
%pwd

'/mnt/d/resume_projects/flight_fare_prediction/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/mnt/d/resume_projects/flight_fare_prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass
class ModelTrainerConfig:
    root_dir: Path
    trained_models_dir: Path
    model_file_name: Path
    expected_score: float
    over_fitting_underfitting_threshold: float


In [7]:
#updating configuration manager
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.entity.config_entity import ModelTrainerConfig

import sys
import os
from src.flight_price_prediction.utils.common import *

In [8]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.model_trainer

        model_trainer_dir = Path(config.root_dir)
        trained_models_dir = Path(config.trained_models_dir)
        model_file_name = Path(config.model_file_name)
        expected_score = params.expected_score
        over_fitting_underfitting_threshold = params.over_fitting_underfitting_threshold

        create_directories([model_trainer_dir,trained_models_dir.parent,model_file_name.parent])

        return ModelTrainerConfig(root_dir = model_trainer_dir,
        trained_models_dir = trained_models_dir,
        model_file_name = model_file_name ,
        expected_score = expected_score,
        over_fitting_underfitting_threshold = over_fitting_underfitting_threshold

        )


In [9]:
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.entity.config_entity import ModelTrainerConfig
from src.flight_price_prediction.entity.artifact_entity import ModelTrainerArtifact,DataTransformationArtifact
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.utils.metrics.regression_metrics import get_regression_score
from src.flight_price_prediction.utils.common import load_bin,save_bin,evaluate_models

from src.flight_price_prediction.utils.ml_utils.model import Flight_Price_Prediction_Model


In [10]:
#model trainer component
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (AdaBoostRegressor,GradientBoostingRegressor,RandomForestRegressor)
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import mlflow
from urllib.parse import urlparse



import dagshub
dagshub.init(repo_owner = 'pavani96-ai', repo_name = 'flight_price_prediction', mlflow=True)

os.environ['MLFLOW_TRACKING_URI'] = os.getenv('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv('MLFLOW_TRACKING_USERNAME')
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv('MLFLOW_TRACKING_PASSWORD')

class ModelTrainer:
    def __init__(self, model_trainer_config: ModelTrainerConfig,
                 data_transformation_artifact: DataTransformationArtifact):
        try:
            self.model_trainer_config = model_trainer_config
            self.data_transformation_artifact = data_transformation_artifact
        except Exception as e:
            raise CustomException(e, sys)
        
    def track_mlflow(self, best_model, regressionmetric,best_model_name):
        mlflow.set_registry_uri(os.getenv('MLFLOw_TRACKING_URI'))
        tracking_url_type_store = urlparse(os.getenv('MLFLOW_TRACKING_URI')).scheme
        with mlflow.start_run():
            r2_score = regressionmetric.r2_score
            mae = regressionmetric.mae
            mse = regressionmetric.mse
            adj_r2_score = regressionmetric.adj_r2_score

            mlflow.log_metric('r2_score', r2_score)
            mlflow.log_metric('mae', mae)
            mlflow.log_metric('mse', mse)
            mlflow.log_metric('adj_r2_score', adj_r2_score)

            if tracking_url_type_store != 'file':
                mlflow.sklearn.log_model(best_model, 'model', registered_model_name=best_model_name)
            else:
                mlflow.sklearn.log_model(best_model, 'model')
    
    def train_model(self,x_train,y_train,x_test,y_test):
        try:
            models = {
                'Linear Regression': LinearRegression(),
                'KNN Regressor': KNeighborsRegressor(),
                'Decision Tree Regressor': DecisionTreeRegressor(),
                'Random Forest Regressor': RandomForestRegressor(),
                'AdaBoost Regressor': AdaBoostRegressor(),
                'Gradient Boosting Regressor': GradientBoostingRegressor(),
                'XGBoost Regressor': XGBRegressor(),
                'CatBoost Regressor': CatBoostRegressor(),
                'LightGBM Regressor': LGBMRegressor()
             }
            params = {
                'Linear Regression': {},
                'KNN Regressor': {
                    'n_neighbors': [3, 5, 7],
                    'weights': ['uniform', 'distance'],
                    'metric': ['euclidean', 'manhattan'],
                    },
                'Decision Tree Regressor': {
                    'max_depth': [None, 10, 20],
                    'min_samples_split': [2, 5, 10],
                    'min_samples_leaf': [1, 2, 4],
                    'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
                    'splitter': ['best', 'random'],
                    'max_features': ['auto', 'sqrt', 'log2', None],
                    'max_leaf_nodes': [None, 5, 10, 15],
                    'ccp_alpha': [0.0, 0.01, 0.1],
                    },
                'Random Forest Regressor': {
                    'n_estimators': [8,16,32,128,256],
                    'max_depth': [None, 10, 20, 30],
                    'min_samples_split':[2, 5, 10],
                    'min_samples_leaf':[1,2,3,4,5],
                    'max_features': ['auto', 'sqrt' , 'log2', None],
                    'criterion':['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
                    'max_leaf_nodes': [None, 5, 10, 15],
                    'ccp_alpha': [0.0, 0.01, 0.1],
                    'bootstrap': [True, False],
                    'oob_score': [True, False],
                    },
                'AdaBoost Regressor': {
                    'n_estimators': [25,32,64,100,128], 
                    'learning_rate': [0.01, 0.1, 0.5, 1.0],
                    'loss': ['linear', 'square', 'exponential'],
                    'random_state': [42, None],
                    'max_depth': [3, 5, 10],
                    'min_samples_split': [2, 5, 10],
                    'estimator__max_depth': [1, 3, 5],
                    'estimator__min_samples_leaf': [1, 2, 4],
                    },
                'Gradient Boosting Regressor': {
                    'loss': ['log_loss','exponential'],
                    'n_estimators': [8,16,32,128,256],
                    'learning_rate': [0.01, 0.1],
                    'criterion': ['squared_error', 'friedman_mse'],
                    'max_features': ['auto', 'sqrt', 'log2', None],
                    'subsample': [0.6,0.7,0.75,0.85,0.9,1.0],

                    },
                'XGBoost Regressor': {
                    'n_estimators': [8,16,32,128,256],
                    'learning_rate': [0.01, 0.1],
                    'max_depth': [3, 5, 10],
                    'subsample': [0.6,0.7,0.75,0.85,0.9,1.0],
                    'colsample_bytree': [0.6,0.7,0.75,0.85,0.9],
                    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
                    'reg_alpha': [0,0.01, 0.1, 0.5, 1.0],
                    'reg_lambda': [0, 0.01, 0.1, 0.5, 1.0],
                    },
                'CatBoost Regressor': {
                    'iterations': [50, 100, 150, 200],
                    'learning_rate': [0.01, 0.1],
                    'depth': [6, 8],
                    'l2_leaf_reg': [1, 3, 5],
                    'border_count': [32, 64, 128],
                    'random_strength': [0.1,0.5, 1.0],
                    'bagging_temperature': [0.1, 0.5, 1.0],
                    'od_type': ['IncToDec', 'Iter'],},
                'LightGBM Regressor': {
                    'n_estimators': [8,16,32,128,256], 
                    'learning_rate': [0.01, 0.10], 
                    'num_leaves': [31, 50],
                    'max_depth': [-1, 1, 5, 10],
                    'min_data_in_leaf': [20, 50, 100],
                    'feature_fraction': [0.6, 0.8, 1.0],
                    'bagging_fraction': [0.6, 0.8, 1.0]},}
            model_report:dict=evaluate_models(x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, models=models, param =params)

            ## To get best model score from dict
            best_model_score = max(sorted(model_report.values()))

            ## To get best model name from dict
            best_model_name = list(model_report.keys())[list(model_report.values()).index(best_model_score)]
            best_model = models[best_model_name]
            y_train_pred = best_model.predict(x_train)
            y_test_pred = best_model.predict(x_test)
            
            #calculationg regression metrics for train set and tracking with mlflow
            regression_train_metric = get_regression_score(y_true=y_train, y_pred=y_train_pred, n_features = x_train.shape[1])
            self.track_mlflow(best_model, regression_train_metric, best_model_name)
            logging.info(f"Best model found on training set: {best_model_name} )")

            #claculating regression metrics for test set and tracking with mlflow
            regression_test_metric = get_regression_score(y_true=y_test, y_pred=y_test_pred, n_features = x_test.shape[1])
            self.track_mlflow(best_model, regression_test_metric, best_model_name)
            logging.info(f"Best Model found on test set: {best_model_name})") 

            preprocessor = load_bin(file_path=self.data_transformation_artifact.preprocessor_object_file_name)
            Flight_Fare_Prediction_Model = Flight_Price_Prediction_Model(preprocessor = preprocessor, model = best_model)
            save_bin(self.model_trainer_config.model_file_name, obj = Flight_Fare_Prediction_Model)
            
            #model pusher
            save_bin("final_model/model.pkl", obj = best_model)

            #3 model trainer artifact
            model_trainer_artifact = ModelTrainerArtifact(model_file_name = self.model_trainer_config.model_file_name,
                                                          train_metric_artifact = regression_train_metric,
                                                          test_metric_artifact = regression_test_metric)
        
            logging.info(f"Model trainer artifact: {model_trainer_artifact}")
            return model_trainer_artifact
        except Exception as e:
            raise CustomException(e, sys)
    
    def initiate_model_trainer(self) -> ModelTrainerArtifact:
        try:
            logging.info('loading transformed training and test data')
            train_file_name = self.data_transformation_artifact.transformed_train_file_name
            test_file_name = self.data_transformation_artifact.transformed_test_file_name

            #loading training array annd testing array
            train_arr = load_bin(path = train_file_name)
            test_arr = load_bin(path = test_file_name)

            x_train, y_train, x_test, y_test = (
                train_arr[:,:-1],
                train_arr[:,-1],
                test_arr[:,:-1],
                test_arr[:,-1])
            model_trainer_artifact = self.train_model(x_train, y_train, x_test, y_test)
            return model_trainer_artifact
        except Exception as e:
            raise CustomException(e, sys)
        

/mnt/d/resume_projects/flight_fare_prediction/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-11 16:31:26,239: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as pavani96-ai

[2026-06-11 16:31:26,275: INFO: helpers: Accessing as pavani96-ai]
[2026-06-11 16:31:27,509: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/pavani96-ai/flight_price_prediction "HTTP/1.1 200 OK"]
[2026-06-11 16:31:28,415: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "pavani96-ai/flight_price_prediction"

[2026-06-11 16:31:28,481: INFO: helpers: Initialized MLflow to track repo "pavani96-ai/flight_price_prediction"]


Repository pavani96-ai/flight_price_prediction initialized!

[2026-06-11 16:31:28,485: INFO: helpers: Repository pavani96-ai/flight_price_prediction initialized!]


In [ ]:
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.entity.config_entity import ModelTrainerConfig
from src.flight_price_prediction.entity.artifact_entity import DataTransformationArtifact,ModelTrainerArtifact


from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging

STAGE_NAME = "Model Trainer Stage"

class ModelTrainerPipeline:
    def __init__(self, config: ConfigurationManager, data_transformation_artifact: DataTransformationArtifact):
       try:
           self.config = config
           self.data_transformation_artifact = data_transformation_artifact
       except Exception as e:
           raise CustomException(e, sys)
       
    def initiate_model_training(self) -> ModelTrainerArtifact:
        try:
            logging.info(f">>>> stage {STAGE_NAME} started <<<<")
            model_trainer_config = self.config.get_model_training_config()
            model_trainer = ModelTrainer(model_trainer_config = model_trainer_config, data_transformation_artifact = self.data_transformation_artifact)
            model_trainer_artifact = model_trainer.initiate_model_trainer()
            logging.info(f">>>> stage {STAGE_NAME} completed <<<<")
            return model_trainer_artifact
        except Exception as e:
            raise CustomException(e, sys)
        

if __name__ == "__main__":
    try:
        logging.info(f'>>>> stage {STAGE_NAME} started <<<<')
        config = ConfigurationManager()
        from src.flight_price_prediction.pipeline.feature_engineering_pipeline import FeatureEngineeringTrainingPipeline
        from src.flight_price_prediction.pipeline.data_ingestion_pipeline import DataIngestionTrainingPipeline
        from src.flight_price_prediction.pipeline.data_validation_pipeline import DataValidationTrainingPipeline
        from src.flight_price_prediction.pipeline.data_transformation_pipeline import DataTransformationTrainingPipeline

        ingestion_pipeline = DataIngestionTrainingPipeline(config=config)
        ingestion_artifact = ingestion_pipeline.initiate_data_ingestion()
        validation_pipeline = DataValidationTrainingPipeline(config=config, data_ingestion_artifact=ingestion_artifact)
        validation_artifact = validation_pipeline.initiate_data_validation()
        feature_engineering_pipeline = FeatureEngineeringTrainingPipeline(config=config, data_validation_artifact=validation_artifact)
        fe_artifact = feature_engineering_pipeline.initiate_feature_engineering()
        transformation_pipeline = DataTransformationTrainingPipeline(config=config, feature_engineering_artifact=fe_artifact)
        transformation_pipeline.initiate_data_transformation()
        model_trainer_pipeline = ModelTrainerPipeline(config=config, data_transformation_artifact=transformation_pipeline.initiate_data_transformation())
        model_trainer_artifact = model_trainer_pipeline.initiate_model_training()
        logging.info(f'>>>> stage {STAGE_NAME} completed <<<<')
    except Exception as e:
        raise CustomException(e, sys)

[2026-06-11 16:31:28,513: INFO: 3488495150: >>>> stage Model Trainer Stage started <<<<]
[2026-06-11 16:31:28,523: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/config/config.yaml loaded succesfully ]
[2026-06-11 16:31:28,538: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/params/params.yaml loaded succesfully ]
[2026-06-11 16:31:28,549: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/schema/schema.yaml loaded succesfully ]
[2026-06-11 16:31:28,559: INFO: common: created directory at: artifacts]
[2026-06-11 16:31:29,686: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-11 16:31:30,096: INFO: data_ingestion: Connecting to MongoDB at: mongodb+srv://p...]
[2026-06-11 16:31:30,952: INFO: data_ingestion: Successfully connected to MongoDB.]
[2026-06-11 16:31:44,741: INFO: common: Data saved to: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-11 16:31:44,742: INFO: data_ingestion: sa